# 04 · Optimizers on FashionMNIST (MLP)

SGD vs momentum vs RMSProp on a small FashionMNIST subset.

Source: `Model_Optimization_Training_Exercises` part 1

**Note:** the course template adds Softmax before `CrossEntropyLoss`. CE already applies log-softmax internally, so this notebook outputs **logits** only.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Subset: 8k train / 2k test so Colab finishes in a few minutes. Batch size 32.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_subset_size, test_subset_size = 8000, 2000
train_subset, _ = random_split(train_dataset, [train_subset_size, len(train_dataset) - train_subset_size],
                               generator=torch.Generator().manual_seed(42))
test_subset, _ = random_split(test_dataset, [test_subset_size, len(test_dataset) - test_subset_size],
                              generator=torch.Generator().manual_seed(42))

batch_size = 32
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)
plt.figure(figsize=(2, 2))
plt.imshow(images[0].squeeze(), cmap='gray')
plt.title(f'label {labels[0].item()}')
plt.axis('off')
plt.show()

In [ ]:
class FashionNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),  # logits — no Softmax
        )

    def forward(self, x):
        return self.network(x)


def run_optimizer(name, make_opt, epochs=8):
    model = FashionNN().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_opt(model)
    history = []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            running += loss.item()
        avg = running / len(train_loader)
        history.append(avg)
        print(f'{name}  epoch {epoch+1}/{epochs}  loss {avg:.4f}')

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            pred = model(images).argmax(dim=1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    acc = 100 * correct / total
    print(f'Accuracy with {name}: {acc:.2f}%')
    return model, history, acc


def show_grid(model, title):
    model.eval()
    images, labels = next(iter(test_loader))
    images_d = images.to(device)
    with torch.no_grad():
        pred = model(images_d).argmax(dim=1).cpu()
    plt.figure(figsize=(8, 8))
    plt.suptitle(title)
    for i in range(25):
        ax = plt.subplot(5, 5, i + 1)
        ax.imshow(images[i].squeeze(), cmap='gray')
        ax.set_title(f'P {pred[i].item()}  T {labels[i].item()}', fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### SGD  lr=0.01  momentum=0

In [ ]:
sgd_model, sgd_hist, sgd_acc = run_optimizer(
    'SGD', lambda m: optim.SGD(m.parameters(), lr=0.01, momentum=0.0)
)
show_grid(sgd_model, 'SGD predictions')

### SGD + momentum  lr=0.01  momentum=0.9

In [ ]:
mom_model, mom_hist, mom_acc = run_optimizer(
    'Momentum', lambda m: optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
)
show_grid(mom_model, 'Momentum predictions')

### RMSProp  lr=0.001  alpha=0.9

In [ ]:
rms_model, rms_hist, rms_acc = run_optimizer(
    'RMSProp', lambda m: optim.RMSprop(m.parameters(), lr=0.001, alpha=0.9)
)
show_grid(rms_model, 'RMSProp predictions')

plt.figure(figsize=(8, 4))
plt.plot(sgd_hist, label=f'SGD ({sgd_acc:.1f}%)')
plt.plot(mom_hist, label=f'Momentum ({mom_acc:.1f}%)')
plt.plot(rms_hist, label=f'RMSProp ({rms_acc:.1f}%)')
plt.xlabel('Epoch')
plt.ylabel('Train loss')
plt.title('Optimizer comparison')
plt.legend()
plt.show()